# BOCD Demo

Run-length-distribution visualization for the FollowThrough Bayesian Online
Changepoint Detector. We feed three synthetic daily-completion-rate streams:

1. **Stable** — uniform 0.85, no changepoint.
2. **Step change** — 0.7 for 20 days, then 0.3 for 20 days.
3. **Transient dip** — 0.85, then 3 bad days, then 0.85 again.

For each, we plot:
- the observed series,
- the changepoint score (`P(r_t ≤ 3)`) over time, and
- the run-length posterior as a heatmap (`log P(r_t | x_{1:t})`).

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from backend.ml.bocd import BOCDDetector, CP_THRESHOLD, SHORT_RUN_K

np.random.seed(0)
plt.rcParams['figure.dpi'] = 110

In [ ]:
def run_detector(series):
    """Replay a series through a fresh detector, returning per-step diagnostics."""
    d = BOCDDetector()
    cp_scores, expected_rls, statuses = [], [], []
    rl_post = []  # list of arrays, one per step
    for y in series:
        cp = d.update(y)
        cp_scores.append(cp)
        expected_rls.append(d.expected_run_length())
        statuses.append(d.get_drift_status())
        rl_post.append(d.run_length_probs.copy())
    return d, np.asarray(cp_scores), np.asarray(expected_rls), statuses, rl_post


def heatmap_matrix(rl_post, T):
    """Stack the per-step run-length posteriors into a (T+1, T) matrix for plotting."""
    M = np.full((T + 1, T), np.nan)
    for t, p in enumerate(rl_post):
        M[: len(p), t] = p
    return M


def plot_run(series, title):
    detector, cps, erl, statuses, rl_post = run_detector(series)
    T = len(series)
    M = heatmap_matrix(rl_post, T)

    fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True,
                              gridspec_kw={'height_ratios': [1, 1, 2]})

    axes[0].plot(series, color='black')
    axes[0].set_ylabel('completion rate')
    axes[0].set_ylim(-0.05, 1.05)
    axes[0].set_title(title)

    axes[1].plot(cps, color='C3', label=f'P(r_t ≤ {SHORT_RUN_K})')
    axes[1].axhline(CP_THRESHOLD, color='gray', ls='--', lw=0.7, label='flag threshold')
    axes[1].set_ylabel('cp score')
    axes[1].set_ylim(-0.05, 1.05)
    axes[1].legend(loc='upper right', fontsize=8)

    im = axes[2].imshow(np.log(M + 1e-9), aspect='auto', origin='lower',
                         cmap='viridis', interpolation='nearest')
    axes[2].set_ylabel('run length r')
    axes[2].set_xlabel('day')
    fig.colorbar(im, ax=axes[2], label='log P(r_t | x)')

    print(f'final drift_status: {detector.get_drift_status()}')
    print(f'final expected run length: {erl[-1]:.2f}')
    plt.tight_layout()
    plt.show()

## 1. Stable series

Uniform 0.85 with mild noise. The run-length posterior should grow steadily; cp
score should hover near zero after warmup.

In [ ]:
stable = np.clip(0.85 + 0.05 * np.random.randn(40), 0, 1)
plot_run(stable, 'Stable regime')

## 2. Step change

20 days at 0.7, then 20 days at 0.3. The cp score should spike past the
threshold within a few days of the step, and the run-length posterior should
visibly restart from r = 0 after day 20.

In [ ]:
step = np.concatenate([
    np.clip(0.7 + 0.05 * np.random.randn(20), 0, 1),
    np.clip(0.3 + 0.05 * np.random.randn(20), 0, 1),
])
plot_run(step, 'Step change at day 20')

## 3. Transient dip

Three bad days flanked by stable performance. The detector flags briefly but
should classify the event as transient, returning to `stable` once the user
recovers.

In [ ]:
transient = np.concatenate([
    np.clip(0.85 + 0.05 * np.random.randn(15), 0, 1),
    np.clip(0.20 + 0.05 * np.random.randn(3), 0, 1),
    np.clip(0.85 + 0.05 * np.random.randn(15), 0, 1),
])
plot_run(transient, 'Transient dip (days 16-18)')

## 4. Serialization round-trip sanity check

Demonstrates that `to_dict` → `from_dict` reconstructs a detector that produces
the identical next observation, supporting persistence across server restarts.

In [ ]:
d = BOCDDetector()
for y in [0.7, 0.6, 0.8, 0.5, 0.9]:
    d.update(y)

state = d.to_dict()
restored = BOCDDetector.from_dict(state)

next_y = 0.4
print(f'cp(orig)     = {d.update(next_y):.6f}')
print(f'cp(restored) = {restored.update(next_y):.6f}')
print('drift_status:', restored.get_drift_status())